# Model Trainging – Heart Disease Dataset
*Training and evaluation of models for Cardio - Risk Prediction project*  

---

## Table of Contents


<a id='imports'></a>
## Reproducibility & Imports  
---

In [26]:
# Reproducibility
import os, sys
import pandas as pd
import joblib

# Imports
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))

# Models & tools
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from scripts.modeltraining.utils import evaluate_classification

SEED = 42

In [27]:
df = pd.read_csv('../data/df_mix.csv')
df.head()

,PCA_1,tSNE_1,UMAP_1,DEATH_EVENT
0,0.198919,-2.080552,5.279534,0
1,1.188811,-3.854919,6.806034,0
2,-0.825730,0.951819,4.962786,0
3,0.408272,-2.558243,5.223496,0
4,1.784139,-4.937725,5.568652,0


In [29]:
df_copy = df.copy()
TARGET_COL = 'DEATH_EVENT'

X = df_copy.drop(columns=[TARGET_COL])
y = df_copy[TARGET_COL]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

Data split

In [30]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Split shapes:")
print("  X_tr:", X_tr.shape, "| X_te:", X_te.shape)

Split shapes:
  X_tr: (246, 3) | X_te: (62, 3)


<a id='random-forest'></a>
---
## Random Forest

In [35]:
params_grid = {
    'n_estimators': [700, 900, 1000, 1200],
    'max_depth': [6, 7, 8, 9],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 0.4, 0.6, 0.8],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced'],
}

In [ ]:
grid = GridSearchCV(
    estimator = RandomForestClassifier(n_jobs=-1, random_state=SEED),
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print('Best parameters:', grid.best_params_)
df_metrics, details = evaluate_classification(model, X_te, y_te)

# save the best model
os.makedirs('../outputs/models', exist_ok=True)
joblib.dump(model, '../outputs/models/rf.joblib')

Fitting 5 folds for each of 2304 candidates, totalling 11520 fits


<a id='adaboost'></a>
## AdaBoost

In [33]:
params_grid = {
    'n_estimators': [50, 100, 200, 600],
    'learning_rate': [0.01, 0.05, 0.1],
    'estimator__max_depth': [1, 2, 3, 4],
    'estimator__min_samples_leaf': [1, 2, 3],
    'estimator__class_weight': [None, 'balanced'],
}

In [34]:
grid = GridSearchCV(
    estimator = AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=SEED), random_state=SEED),
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print('Best params:', grid.best_params_)
df_metrics, details = evaluate_classification(model, X_te, y_te)

os.makedirs('../outputs/models', exist_ok=True)
joblib.dump(model, '../outputs/models/ada.joblib')

Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Best params: {'estimator__class_weight': None, 'estimator__max_depth': 3, 'estimator__min_samples_leaf': 3, 'learning_rate': 0.05, 'n_estimators': 200}


,Metric,Value
0,Accuracy,0.9032
1,Balanced accuracy,0.8920
2,Precision (pos=1),0.8571
3,Recall / Sensitivity (TPR),0.8571
4,Specificity (TNR),0.9268
5,F1,0.8571
6,F0.5,0.8571
7,F2,0.8571
8,ROC-AUC,0.9547
9,PR-AUC (Average Precision),0.9036



Confusion matrix:


,Pred 0,Pred 1
Actual 0,38,3
Actual 1,3,18


['../outputs/models/ada.joblib']

<a id='xgboost'></a>
## XGBoost


In [31]:
params_grid = {
    'n_estimators': [35, 40, 50, 100, 150],   
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4, 5, 6],
    'min_child_weight': [1, 2, 3, 4, 5, 6],
    'subsample': [0.5, 1.0],
    'reg_lambda': [0.01, 0.05, 0.5, 0.1, 4.0, 5.0],
}


In [32]:
xgb = XGBClassifier(
    tree_method = 'hist',
    objective = 'binary:logistic',
    eval_metric = 'logloss',
    n_jobs = -1,
    random_state = SEED,
)

grid = GridSearchCV(
    estimator = xgb,
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
model = grid.best_estimator_

print('Best params:', grid.best_params_)
df_metrics, details = evaluate_classification(model, X_te, y_te)

os.makedirs('../outputs/models', exist_ok=True)
joblib.dump(model, '../outputs/models/xgb.joblib')

Fitting 5 folds for each of 5400 candidates, totalling 27000 fits
Best params: {'learning_rate': 0.05, 'max_depth': 2, 'min_child_weight': 6, 'n_estimators': 150, 'reg_lambda': 0.5, 'subsample': 1.0}


,Metric,Value
0,Accuracy,0.9032
1,Balanced accuracy,0.8920
2,Precision (pos=1),0.8571
3,Recall / Sensitivity (TPR),0.8571
4,Specificity (TNR),0.9268
5,F1,0.8571
6,F0.5,0.8571
7,F2,0.8571
8,ROC-AUC,0.9355
9,PR-AUC (Average Precision),0.7901



Confusion matrix:


,Pred 0,Pred 1
Actual 0,38,3
Actual 1,3,18


['../outputs/models/xgb.joblib']